In [34]:
# Importing torch
import numpy as np
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.13.0+cu126
0.28.0+cu126


In [35]:
# GPU avaliability
if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU.")

GPU is available!
Using GPU: NVIDIA GeForce MX350


#### Tensor Creation Methods

In [36]:
# This method will assign a memory to create a tensor of provided shape and return the existing numbers at that location
a = torch.empty(size=[2, 3], dtype=torch.float) # default float32
a

tensor([[8.8947e-09, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 6.5920e-10, 0.0000e+00]])

Scaler (1D) → Vector (2D) → Tensor (> 2D)

In [37]:
# Check type
type(a)

torch.Tensor

In [38]:
# Check the memory consuption
from sys import getsizeof

print(f"Total Memory {getsizeof(a)}")
print(f"Tensor Memory {a.nbytes}") # Float of 4 bytes

# Total Memory → 6(total numbers) * [4(size of float32) + 8(size of pointer)] = 72

Total Memory 72
Tensor Memory 24


*The core difference is that sys.getsizeof() measures the total memory footprint of an object (including overhead and references), while .nbytes measures only the raw data size of the object's elements, excluding overhead.*

In [39]:
# Using zeros
torch.zeros(size = [2, 3], dtype = torch.float32)

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [40]:
# using ones
torch.ones(size = [2, 3])

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [41]:
# using rand
torch.rand(size = [2, 3])

tensor([[0.1117, 0.7477, 0.8505],
        [0.0562, 0.2325, 0.1522]])

In [42]:
# manual_seed - for regenerating the same array
seed_number = torch.initial_seed() # Store this seed number and use it as a variable → python long datatype
print(seed_number)
# NOTE: seed number act as a starting point to generate the sequences and numbers avalable on those locations will be in the resultant tensor

torch.manual_seed(seed_number)
torch.rand(2,3)

10965798010643897182


tensor([[0.1117, 0.7477, 0.8505],
        [0.0562, 0.2325, 0.1522]])

In [43]:
# Device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [44]:
# using tensor

torch.tensor(
    data = [[1,2,3],[4,5,6]],
    dtype = torch.float32,
    requires_grad = False,
    device = device,
    pin_memory = True # The tensor will directly loded into pin-memory and not pager-memory, for faster loading. pin_memory=True is primarily a GPU optimization tool that prepares CPU data for faster transfer to the GPU.
)

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')

In [45]:
# NOTE: You can only calculate the gradient when the end result (loss function) is scaler.
try:
    a = torch.rand(size=[2, 3], requires_grad=True, pin_memory=True).to(device=device)
    b = a.pow(exponent=2)
    c = b * 2 # c = (a ^ 2) + 2

    c.backward()

except RuntimeError as e:
    print(e)

grad can be implicitly created only for scalar outputs


In [46]:
# Question: Which one is fast?
torch.rand(size=[2, 3]).to(device="cuda") # Normal
torch.rand(size=[2, 3], pin_memory=True).to(device="cuda") # Slow: CPU (pager memory) → CPU (pin memory) → VRAM(GPU memory)
torch.rand(size=[2, 3], device="cuda") # Fast

tensor([[0.7143, 0.2458, 0.5469],
        [0.8111, 0.9936, 0.0805]], device='cuda:0')

[A guide on good usage of non_blocking and pin_memory() in PyTorch](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)

In [47]:
# arange
print("using arange ->", torch.arange(start=0, end=10, step=2))

# using linspace
print("using linspace ->", torch.linspace(start=0, end=10, steps=10))

# using eye - Identity Matrix tensor
print("using eye ->", torch.eye(n=5))

# using full - torch.ones(size = (m, n)) * constant( = 5)
print("using full ->", torch.full(size=(3, 3), fill_value=5))

using arange -> tensor([0, 2, 4, 6, 8])
using linspace -> tensor([ 0.0000,  1.1111,  2.2222,  3.3333,  4.4444,  5.5556,  6.6667,  7.7778,
         8.8889, 10.0000])
using eye -> tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
using full -> tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]])


---

#### Tensor Shape

In [48]:
x = torch.tensor([[1,2,3],[4,5,6]])
x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [49]:
print(x.shape)
print(x.size(dim=1)) # You can pass the dimension as well → dim = 0 (rows - outermost dimension)

torch.Size([2, 3])
3


In [50]:
# Create an uninitialized tensor with the same shape as the input tensor
y = torch.empty_like(
    input=x,
    memory_format=torch.preserve_format, # The individual elements in this tensor pointing to the same location where the elements of first tensor is pointing -> else some random numbers with equal shape 
    dtype=torch.float32,
    device=device,
    pin_memory=False,
    requires_grad=True
)

print(y)

tensor([[0.7143, 0.2458, 0.5469],
        [0.8111, 0.9936, 0.0805]], device='cuda:0', requires_grad=True)


In [33]:
a = torch.rand(size=[5], requires_grad=True)
b = torch.ones_like(input=a, memory_format=torch.preserve_format)
print(b.requires_grad) # no gradient computations by default

print(id(a[1]), id(b[1]))
print(a[1].item(), b[1].item()) 
# temporial behavour of python's id() function for numpy arrays and pytorch tensors explination → /Notes/id method & Torch Memory Format

b[0] = 32
print(a[0].item())

False
139182143694800 139182143694800
0.9532780647277832 1.0
0.20786720514297485


In [51]:
# torch.preserve_format
print(id(x[0][0]))
print(id(y[0][0]))
# NOTE: Since my device is changed (x - cpu) and (y - gpu) the memory locations of elements are different.

134791740364368
134791740364496


In [52]:
torch.zeros_like(x)

tensor([[0, 0, 0],
        [0, 0, 0]])

In [53]:
torch.ones_like(x)

tensor([[1, 1, 1],
        [1, 1, 1]])

In [54]:
torch.rand_like(x, dtype=torch.float32)

tensor([[0.9235, 0.8624, 0.4057],
        [0.8779, 0.1056, 0.5866]])

*`_like()` These methods create a new tensor with the exact same shape, dtype, and device as an existing input tensor but not `gradient computations`.*

---

#### Tensor DataTypes

In [55]:
# find data type
x.dtype

torch.int64

In [56]:
# assign data type
torch.tensor([1.0, 2.0, 3.0], dtype = torch.int32)

tensor([1, 2, 3], dtype=torch.int32)

In [57]:
torch.tensor([1, 2, 3], dtype = torch.float64)

tensor([1., 2., 3.], dtype=torch.float64)

In [58]:
# using to() - Type Conversion - Generally used to transfer a tensor from cpu to gpu
x.to(torch.float32)

tensor([[1., 2., 3.],
        [4., 5., 6.]])

| **Data Type**             | **Dtype**         | **Description**                                                                                                                                                                |
|---------------------------|-------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **32-bit Floating Point** | `torch.float32`   | Standard floating-point type used for most deep learning tasks. Provides a balance between precision and memory usage.                                                         |
| **64-bit Floating Point** | `torch.float64`   | Double-precision floating point. Useful for high-precision numerical tasks but uses more memory.                                                                               |
| **16-bit Floating Point** | `torch.float16`   | Half-precision floating point. Commonly used in mixed-precision training to reduce memory and computational overhead on modern GPUs.                                            |
| **BFloat16**              | `torch.bfloat16`  | Brain floating-point format with reduced precision compared to `float16`. Used in mixed-precision training, especially on TPUs.                                                |
| **8-bit Floating Point**  | `torch.float8`    | Ultra-low-precision floating point. Used for experimental applications and extreme memory-constrained environments (less common).                                               |
| **8-bit Integer**         | `torch.int8`      | 8-bit signed integer. Used for quantized models to save memory and computation in inference.                                                                                   |
| **16-bit Integer**        | `torch.int16`     | 16-bit signed integer. Useful for special numerical tasks requiring intermediate precision.                                                                                    |
| **32-bit Integer**        | `torch.int32`     | Standard signed integer type. Commonly used for indexing and general-purpose numerical tasks.                                                                                  |
| **64-bit Integer**        | `torch.int64`     | Long integer type. Often used for large indexing arrays or for tasks involving large numbers.                                                                                  |
| **8-bit Unsigned Integer**| `torch.uint8`     | 8-bit unsigned integer. Commonly used for image data (e.g., pixel values between 0 and 255).                                                                                    |
| **Boolean**               | `torch.bool`      | Boolean type, stores `True` or `False` values. Often used for masks in logical operations.                                                                                      |
| **Complex 64**            | `torch.complex64` | Complex number type with 32-bit real and 32-bit imaginary parts. Used for scientific and signal processing tasks.                                                               |
| **Complex 128**           | `torch.complex128`| Complex number type with 64-bit real and 64-bit imaginary parts. Offers higher precision but uses more memory.                                                                 |
| **Quantized Integer**     | `torch.qint8`     | Quantized signed 8-bit integer. Used in quantized models for efficient inference.                                                                                              |
| **Quantized Unsigned Integer** | `torch.quint8` | Quantized unsigned 8-bit integer. Often used for quantized tensors in image-related tasks.                                                                                     |


---

#### Mathematical Operations

In [59]:
x = torch.rand(2,2)
x

tensor([[0.6719, 0.8896],
        [0.7094, 0.7251]])

In [60]:
# Scalar Operations

# addition
print(x + 2, end = "\n\n")

# substraction
print(x - 2, end = "\n\n")

# multiplication
print(x * 3, end = "\n\n")

# division
print(x / 3, end = "\n\n")

# int division
print((x * 100) // 3, end = "\n\n")

# mod
print(((x * 100) // 3) % 2, end = "\n\n")

# power
print(x ** 2)

tensor([[2.6719, 2.8896],
        [2.7094, 2.7251]])

tensor([[-1.3281, -1.1104],
        [-1.2906, -1.2749]])

tensor([[2.0156, 2.6689],
        [2.1283, 2.1754]])

tensor([[0.2240, 0.2965],
        [0.2365, 0.2417]])

tensor([[22., 29.],
        [23., 24.]])

tensor([[0., 1.],
        [1., 0.]])

tensor([[0.4514, 0.7914],
        [0.5033, 0.5258]])


In [61]:
# floor division
(torch.tensor([1.0, 2.0, 3.0]) * 100) // 3

tensor([ 33.,  66., 100.])

In [62]:
# Element wise operation
a = torch.rand(2,3)
b = torch.rand(2,3)

print(a)
print(b)

tensor([[0.1140, 0.7693, 0.9795],
        [0.9488, 0.6076, 0.6928]])
tensor([[0.7717, 0.1629, 0.8226],
        [0.1946, 0.8188, 0.8342]])


In [63]:
# Shape must be same or able to broadcast

# add
print(a + b, end="\n\n")

# sub
print(a - b, end="\n\n")

# multiply
print(a * b, end="\n\n")

# division
print(a / b, end="\n\n")

# power
print(a ** b, end="\n\n")

# mod
print(a % b)

tensor([[0.8856, 0.9322, 1.8021],
        [1.1435, 1.4264, 1.5270]])

tensor([[-0.6577,  0.6064,  0.1570],
        [ 0.7542, -0.2111, -0.1413]])

tensor([[0.0879, 0.1253, 0.8057],
        [0.1847, 0.4975, 0.5779]])

tensor([[0.1477, 4.7231, 1.1908],
        [4.8749, 0.7421, 0.8306]])

tensor([[0.1871, 0.9582, 0.9831],
        [0.9898, 0.6651, 0.7363]])

tensor([[0.1140, 0.1178, 0.1570],
        [0.1703, 0.6076, 0.6928]])


In [64]:
c = torch.tensor([1, -2, 3, -4])
c

tensor([ 1, -2,  3, -4])

In [65]:
# abs
torch.abs(c)

tensor([1, 2, 3, 4])

In [66]:
# negative
torch.neg(c)

tensor([-1,  2, -3,  4])

In [67]:
# Inplace operations
print(torch.abs_(c))
print(c)

print(torch.neg_(c))
print(c)

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([-1, -2, -3, -4])
tensor([-1, -2, -3, -4])


In [68]:
c.fill_(value=10)
# In-place equivalent → c = torch.full_like(c, fill_value=10)

tensor([10, 10, 10, 10])

In [69]:
x = torch.empty_like(input=c)
c.copy_(other=x) # c gets updated, In-place equivalent → c = x.clone()
print(c)

tensor([              0, 134796208403520,       936693072,               0])


In [70]:
d = torch.tensor([1.9, 2.3, 3.7, 4.4, 6.5])
d

tensor([1.9000, 2.3000, 3.7000, 4.4000, 6.5000])

In [71]:
# round
torch.round(d) # 6.5 -> 6 (get floored not ceiled)

tensor([2., 2., 4., 4., 6.])

In [72]:
# ceil - nearest bigger integer value
torch.ceil(d)

tensor([2., 3., 4., 5., 7.])

In [73]:
# floor - nearest smaller integer value
torch.floor(d)

tensor([1., 2., 3., 4., 6.])

In [74]:
# clamp | clip - Clipping the values in a certing range
torch.clip(d, min = -1, max = 1)
torch.clamp(d, min = -1, max = 1)

tensor([1., 1., 1., 1., 1.])

In [75]:
e = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float32)
e

tensor([[0., 9., 8.],
        [9., 4., 8.]])

In [76]:
# NumPy `axis` vs Pandas `axis` vs PyTorch `dim` → All are same or might change with function
import pandas as pd
import numpy as np
import torch

print(np.array(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0))

print(pd.DataFrame(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0).to_numpy()) # string arguments → `index` = 0 and `columns` = 1

print(torch.tensor(
    [[1, 2, 3], [4, 5, 6]]
).sum(dim=0))

[5 7 9]
[5 7 9]
tensor([5, 7, 9])


In [77]:
# Aggregate operation

# sum
print(torch.sum(e))

# sum along columns
print(torch.sum(e, dim = 0, keepdim=False)) # same as `axis` parameter in numpy

# sum along rows
print(torch.sum(e, dim = 1, keepdim=False))

# keepdim = True
print(torch.sum(e, dim = 1, keepdim=True))

tensor(38.)
tensor([ 9., 13., 16.])
tensor([17., 21.])
tensor([[17.],
        [21.]])


*If `keepdim` is `False`, the output tensor will not retain the dimension across which the sum was performed.*

In [78]:
# mean
torch.mean(e)

# mean along col
torch.mean(e, dim=0)

tensor([4.5000, 6.5000, 8.0000])

In [79]:
# median
torch.median(e)

tensor(8.)

In [80]:
# max and min
torch.max(e)
torch.min(e)

tensor(0.)

In [81]:
# product
torch.prod(e) # .mul() is a element-wise multiplication between two matrics

tensor(0.)

In [82]:
# standard deviation
torch.std(e)

tensor(3.6148)

In [83]:
# variance
torch.var(e)

tensor(13.0667)

In [84]:
# argmax
torch.argmax(e)

tensor(1)

In [85]:
# argmin
torch.argmin(e)

tensor(0)

In [86]:
# More _methods()
a = torch.full_like(input=e, fill_value=10)
b = a.clone().copy_(a) * 100 # the shape of a and b must be same for copy_ method
print(a, b, sep="\n")

# clone and copy_ → both the tensor elements are pointing to the same memory location

a.clone().add_(b) # a = a + b (ignore clone)
a.clone().sub_(b) # a = a - b 
a.clone().mul_(b) # a = a * b (element wise)

a.zero_() # x = torch.zeros_like(x)

tensor([[10., 10., 10.],
        [10., 10., 10.]])
tensor([[1000., 1000., 1000.],
        [1000., 1000., 1000.]])


tensor([[0., 0., 0.],
        [0., 0., 0.]])

---

#### Matrix operations

In [87]:
f = torch.randint(size=(2,3), low=0, high=10)
g = torch.randint(size=(3,2), low=0, high=10)

print(f)
print(g)

tensor([[1, 1, 4],
        [5, 6, 4]])
tensor([[4, 4],
        [8, 5],
        [9, 2]])


In [88]:
# matrix multiplcation - dot product
torch.matmul(f, g)

tensor([[ 48,  17],
        [104,  58]])

In [89]:
vector1 = torch.tensor([1, 2])
vector2 = torch.tensor([3, 4])

# dot product - only works with 1D Tensors
torch.dot(vector1, vector2)

tensor(11)

In [90]:
# transpose
torch.transpose(input=f, dim0=0, dim1=1)

tensor([[1, 5],
        [1, 6],
        [4, 4]])

In [91]:
torch.transpose(f, dim0 = 1, dim1 = 0) # Transpose is same as `reshape` operation by swapping the dimensions

tensor([[1, 5],
        [1, 6],
        [4, 4]])

In [92]:
# 0th axis - 1, 1st axis - 2, 2nd axis - 3 → consider size as a list and axis is an index
torch.rand(size=[1, 2, 3,]).transpose(dim0=0, dim1=2).shape

torch.Size([3, 2, 1])

In [93]:
# Implace Transpose → only works with 2D
f.t_()

tensor([[1, 5],
        [1, 6],
        [4, 4]])

In [94]:
temp = torch.tensor([
    [[1], [2]],
    [[1], [3]],
    [[1], [4]],
])

print(temp.shape)
torch.transpose(temp, dim0=0, dim1=1).shape # Check the dimensions and its index correctly

torch.Size([3, 2, 1])


torch.Size([2, 3, 1])

*Tip: To visualize a transpose, flatten the tensor into a 1D array and then re-arrange the elements according to the target shape.*

*For a tensor with shape [2, 3, 1], the 0th dimension has size 2, the 1st dimension has size 3, and so on.*

In [95]:
# Convert any nD tensor to 1D tensor
print(torch.tensor([[1, 2], [3, 4]]).ravel())
print(torch.tensor([[1, 2], [3, 4]]).flatten())
print(torch.tensor([[[1], [2]], [[3], [4]]]).reshape(-1))

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])


In [96]:
h = torch.randint(size=(3,3), low=0, high=10, dtype=torch.float32)
h

tensor([[5., 5., 6.],
        [4., 5., 8.],
        [4., 9., 0.]])

In [97]:
# determinant
torch.det(h)

tensor(-104.0000)

In [98]:
# inverse
torch.inverse(h)

tensor([[ 0.6923, -0.5192, -0.0962],
        [-0.3077,  0.2308,  0.1538],
        [-0.1538,  0.2404, -0.0481]])

---

#### Comparison operations

In [99]:
i = torch.randint(size=(2,3), low=0, high=10)
j = torch.randint(size=(2,3), low=0, high=10)

print(i)
print(j)

tensor([[6, 8, 9],
        [5, 2, 0]])
tensor([[0, 6, 8],
        [9, 7, 6]])


In [100]:
# Element wise comparison - Shape must be same or able to broadcast

# greater than
print(i > j)

# less than
print(i < j)

# equal to
print(i == j)

# not equal to
print(i != j)

tensor([[ True,  True,  True],
        [False, False, False]])
tensor([[False, False, False],
        [ True,  True,  True]])
tensor([[False, False, False],
        [False, False, False]])
tensor([[True, True, True],
        [True, True, True]])


In [101]:
# Boolean Indexing - Reshaped into 1D
i[i > j]

tensor([6, 8, 9])

---

#### Special functions

In [102]:
k = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float64)
k

tensor([[9., 5., 7.],
        [6., 9., 2.]], dtype=torch.float64)

In [103]:
# log - base e
torch.log(input = k)

# log - base 10
torch.log10(input = k)

tensor([[0.9542, 0.6990, 0.8451],
        [0.7782, 0.9542, 0.3010]], dtype=torch.float64)

In [104]:
# Checking the size of the elements in the tensor
tensor = torch.log10(input = k)
print(tensor.dtype)
tensor.element_size()  # size in bytes of a single element

torch.float64


8

In [105]:
# exp
torch.exp(k)

tensor([[8.1031e+03, 1.4841e+02, 1.0966e+03],
        [4.0343e+02, 8.1031e+03, 7.3891e+00]], dtype=torch.float64)

In [106]:
# sqrt
torch.sqrt(k)

tensor([[3.0000, 2.2361, 2.6458],
        [2.4495, 3.0000, 1.4142]], dtype=torch.float64)

In [107]:
# sigmoid
torch.sigmoid(k)

tensor([[0.9999, 0.9933, 0.9991],
        [0.9975, 0.9999, 0.8808]], dtype=torch.float64)

In [108]:
# softmax
torch.softmax(k, dim = 1)

tensor([[8.6681e-01, 1.5876e-02, 1.1731e-01],
        [4.7385e-02, 9.5175e-01, 8.6788e-04]], dtype=torch.float64)

In [109]:
# relu
torch.relu(k)

tensor([[9., 5., 7.],
        [6., 9., 2.]], dtype=torch.float64)

---

#### Inplace Operations

In [110]:
m = torch.rand(2,3)
n = torch.rand(2,3)

print(m)
print(n)

tensor([[0.6550, 0.1340, 0.7150],
        [0.2514, 0.8728, 0.4432]])
tensor([[0.2795, 0.9691, 0.6695],
        [0.0017, 0.9699, 0.5412]])


In [111]:
# _ methods
m.add_(n) # It returns the new matrix

tensor([[0.9344, 1.1030, 1.3845],
        [0.2531, 1.8426, 0.9844]])

In [112]:
m # Updated

tensor([[0.9344, 1.1030, 1.3845],
        [0.2531, 1.8426, 0.9844]])

In [113]:
n

tensor([[0.2795, 0.9691, 0.6695],
        [0.0017, 0.9699, 0.5412]])

In [114]:
torch.relu(m)

tensor([[0.9344, 1.1030, 1.3845],
        [0.2531, 1.8426, 0.9844]])

In [115]:
m.relu_()

tensor([[0.9344, 1.1030, 1.3845],
        [0.2531, 1.8426, 0.9844]])

In [116]:
m

tensor([[0.9344, 1.1030, 1.3845],
        [0.2531, 1.8426, 0.9844]])

---

#### Copying a Tensor

In [117]:
a = torch.tensor(
    data = [[1,2,3], [4,5,6]]
)
b = a.clone()

print(a, b, sep = '\n')

print(id(a), id(b))
print(id(a[0,0]), id(b[0][0]))

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])
134791740514128 134791740512336
134791740515920 134791740510928


In [118]:
a[0][0] = 10

print(a, b, sep = '\n')

tensor([[10,  2,  3],
        [ 4,  5,  6]])
tensor([[1, 2, 3],
        [4, 5, 6]])


---

A third-order polynomial, trained to predict `y = sin(x) from -pi to pi` by minimizing squared Euclidean distance.

In [119]:
# Variables Initialization
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float
learning_rate = 1e-6

# Dataset
x = torch.linspace(start=-torch.pi, end=torch.pi, steps=2000) # shape -> 2000 1D
y = torch.sin(input=x) # Shape -> 2000 1D

# Random Weights → for 3 degree term | shape -> Scaler
a = torch.randn((), dtype=dtype)
b = torch.randn((), dtype=dtype)
c = torch.randn((), dtype=dtype)
d = torch.randn((), dtype=dtype)

def y_pred_equation(X_train):
    y_pred = a + (b * x) + (c * x.pow(2)) + (d * x.pow(3))
    return y_pred # shape -> 2000 1D

def main():
    # Training
    for i in range(2500):
        # Prediction
        y_pred = y_pred_equation(x)

        # Loss Calculation
        loss = (y - y_pred).pow(2).sum().item()

        if i % 100 == 99:
            print(f"loss at {i + 1}it epoch → {loss}")

        # Gradients Calculation
        grad_y_pred = -2 * (y - y_pred) # shape -> 2000
        grad_a = grad_y_pred.sum()
        grad_b = torch.matmul(input = grad_y_pred, other = x).sum()
        grad_c = torch.matmul(input = grad_y_pred, other = x.pow(2)).sum()
        grad_d = torch.matmul(input = grad_y_pred, other = x.pow(3)).sum()

        # Updating
        global a, b, c, d
        a = a - learning_rate * grad_a
        b -= learning_rate * grad_b
        c -= learning_rate * grad_c
        d -= learning_rate * grad_d

main()

loss at 100it epoch → 3804.40966796875
loss at 200it epoch → 2531.68408203125
loss at 300it epoch → 1686.284423828125
loss at 400it epoch → 1124.5687255859375
loss at 500it epoch → 751.2260131835938
loss at 600it epoch → 503.0030517578125
loss at 700it epoch → 337.9102478027344
loss at 800it epoch → 228.06649780273438
loss at 900it epoch → 154.95387268066406
loss at 1000it epoch → 106.26966857910156
loss at 1100it epoch → 73.83744049072266
loss at 1200it epoch → 52.22228240966797
loss at 1300it epoch → 37.80915069580078
loss at 1400it epoch → 28.193618774414062
loss at 1500it epoch → 21.77532196044922
loss at 1600it epoch → 17.488689422607422
loss at 1700it epoch → 14.62399673461914
loss at 1800it epoch → 12.708467483520508
loss at 1900it epoch → 11.426745414733887
loss at 2000it epoch → 10.568562507629395
loss at 2100it epoch → 9.99354362487793
loss at 2200it epoch → 9.607961654663086
loss at 2300it epoch → 9.349224090576172
loss at 2400it epoch → 9.175457954406738
loss at 2500it epoc